In [1]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_core.documents import Document
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

/home/vxh/RAG-advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SemanticChunker:
    def __init__(self, embedding_model="all-MiniLM-L6-v2",  similarity_threshold=0.75, 
    min_chunk_size=2,
    max_chunk_size=2000):
        # Tải mô hình embedding nhỏ gọn, tốc độ cao để tính toán nội bộ
        self.model = SentenceTransformer(embedding_model)
        self.threshold = similarity_threshold
        self.min_sentences = min_chunk_size
        self.max_chars = max_chunk_size
    def _split_sentences(self, text):
        """
        Tách câu, xử lý Edge Cases: Giữ nguyên Code Blocks và ghép các câu quá ngắn.
        """
        code_blocks = []
        def replace_code(match):
            code_blocks.append(match.group(0))
            return f" <CODE_BLOCK_{len(code_blocks)-1}> "
        
        text_no_code = re.sub(r'```.*?```', replace_code, text, flags=re.DOTALL)
        
        raw_sentences = re.split(r'(?<=[.!?])\s+', text_no_code)
        
        sentences = []
        for s in raw_sentences:
            s = s.strip()
            if not s: continue
            
            for i, block in enumerate(code_blocks):
                s = s.replace(f"<CODE_BLOCK_{i}>", block)
            
            if len(s.split()) < 5 and sentences:
                sentences[-1] += " " + s
            else:
                sentences.append(s)
                
        return sentences
    def create_chunks(self,text):
        sentences = self._split_sentences(text)
        if not sentences: return []
        chunks=[]
        current_chunk=[sentences[0]]
        current_length=len(sentences[0])
        embeddings = self.model.encode(sentences)
        
        for i in range (1, len(sentences)):
            sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
            sentence=sentences[i]
            sen_len=len(sentence)
            is_below_threshold = sim < self.threshold
            has_min_sentences = len(current_chunk) >= self.min_sentences
            exceeds_max_chars = (current_length + sen_len) > self.max_chars

            if (is_below_threshold and has_min_sentences) or exceeds_max_chars:
                # Đóng gói chunk hiện tại
                chunks.append(" ".join(current_chunk))
                # Khởi tạo chunk mới
                current_chunk = [sentence]
                current_length = sen_len
            else:
                # Nếu hai câu vẫn cùng chủ đề, gộp tiếp vào chunk
                current_chunk.append(sentence)
                current_length += sen_len + 1

        # Push chunk cuối cùng
        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks
    def split_document(self, documents):
        chunked_docs = []
        for doc in documents:
            chunks = self.create_chunks(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunked_docs.append(Document(
                    page_content=chunk,
                    metadata={
                        "source": f"{doc.metadata.get('source', 'unknown')}_chunk_{i}"
                    }
                ))
        return chunked_docs

In [3]:

import os
data_dir = "./data"
documents = []
for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(data_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            doc = Document(page_content=content, metadata={"source": filename})
            documents.append(doc)

print(f"Loaded {len(documents)} documents successfully")

Loaded 3 documents successfully


In [4]:
documents

[Document(metadata={'source': 'doc1.txt'}, page_content='---\nDocument ID: FPT-HR-001\nTitle: Chính sách Thời gian làm việc và Làm thêm giờ\nEffective Date: 01/01/2024\nScope: Áp dụng cho toàn bộ Nhân viên (Employee) và Quản lý (Manager) tại FPT.\n---\n\n## 1. Thời gian làm việc tiêu chuẩn\n* Quy định áp dụng cho tất cả Nhân viên: Thời gian làm việc tiêu chuẩn là 8 giờ/ngày, từ Thứ Hai đến Thứ Sáu.\n* Yêu cầu đối với Quản lý: Quản lý cấp trung trở lên phải đảm bảo có mặt tại văn phòng ít nhất 4 ngày/tuần.\n* Thời gian bắt đầu: Nhân viên phải có mặt và check-in trên hệ thống myFPT trước 08:30 sáng. Đi muộn quá 30 phút sẽ bị trừ 50,000 VND vào lương tháng.\n\n## 2. Quy định Làm thêm giờ (Overtime)\n* Mức chi trả: Làm thêm giờ vào ngày thường sẽ được thanh toán 150% lương cơ bản. Làm thêm giờ vào cuối tuần (Thứ 7, Chủ Nhật) được thanh toán 200%.\n* Điều kiện áp dụng: Việc làm thêm giờ phải được Quản lý phê duyệt trước 17:00 ngày làm việc thông thường.\n* Tham chiếu: Quy trình duyệt Overti

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time
import numpy as np
docs=documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
recursive_spillter = RecursiveCharacterTextSplitter(
                    chunk_size=500, chunk_overlap=50)
start_time = time.time()
recursive_chunks = recursive_spillter.split_documents(docs)
recursive_time = time.time() - start_time

print("1. Recursive Chunking (Cắt theo ký tự)")
print(f"- Thời gian xử lý: {recursive_time:.4f} giây")
print(f"- Số lượng chunks: {len(recursive_chunks)}")
print(f"- Kích thước trung bình: {np.mean([len(c.page_content) for c in recursive_chunks]):.0f} ký tự\n")

# Semantic Chunking
semantic_splitter = SemanticChunker(
    similarity_threshold=0.75, 
    min_chunk_size=2,
    max_chunk_size=2000
)
start_time = time.time()
semantic_chunks = semantic_splitter.split_document(docs)
semantic_time = time.time() - start_time

print("\n2. Semantic Chunking (Cắt theo ngữ nghĩa)")
print(f"- Thời gian xử lý: {semantic_time:.4f} giây")
print(f"- Số lượng chunks: {len(semantic_chunks)}")
print(f"- Kích thước trung bình: {np.mean([len(c.page_content) for c in semantic_chunks]):.0f} ký tự\n")

# --- (VISUAL CHECK) ---
print("--- VÍ DỤ CÁC CHUNKS CỦA SEMANTIC ---")
for i, chunk in enumerate(semantic_chunks):
    print(f"\n[Chunk {i+1}] (Dài {len(chunk.page_content)} ký tự):")
    print("-" * 40)
    print(chunk.page_content)
    print("-" * 40)

1. Recursive Chunking (Cắt theo ký tự)
- Thời gian xử lý: 0.0003 giây
- Số lượng chunks: 9
- Kích thước trung bình: 287 ký tự



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25089.34it/s]



2. Semantic Chunking (Cắt theo ngữ nghĩa)
- Thời gian xử lý: 0.1594 giây
- Số lượng chunks: 12
- Kích thước trung bình: 215 ký tự

--- VÍ DỤ CÁC CHUNKS CỦA SEMANTIC ---

[Chunk 1] (Dài 336 ký tự):
----------------------------------------
---
Document ID: FPT-HR-001
Title: Chính sách Thời gian làm việc và Làm thêm giờ
Effective Date: 01/01/2024
Scope: Áp dụng cho toàn bộ Nhân viên (Employee) và Quản lý (Manager) tại FPT. ---

## 1. Thời gian làm việc tiêu chuẩn
* Quy định áp dụng cho tất cả Nhân viên: Thời gian làm việc tiêu chuẩn là 8 giờ/ngày, từ Thứ Hai đến Thứ Sáu.
----------------------------------------

[Chunk 2] (Dài 200 ký tự):
----------------------------------------
* Yêu cầu đối với Quản lý: Quản lý cấp trung trở lên phải đảm bảo có mặt tại văn phòng ít nhất 4 ngày/tuần. * Thời gian bắt đầu: Nhân viên phải có mặt và check-in trên hệ thống myFPT trước 08:30 sáng.
----------------------------------------

[Chunk 3] (Dài 245 ký tự):
----------------------------------------
Đi 

         

Task 2: Configure HNSW Index

In [6]:
import time
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

In [7]:
semantic_chunks[0]

Document(metadata={'source': 'doc1.txt_chunk_0'}, page_content='---\nDocument ID: FPT-HR-001\nTitle: Chính sách Thời gian làm việc và Làm thêm giờ\nEffective Date: 01/01/2024\nScope: Áp dụng cho toàn bộ Nhân viên (Employee) và Quản lý (Manager) tại FPT. ---\n\n## 1. Thời gian làm việc tiêu chuẩn\n* Quy định áp dụng cho tất cả Nhân viên: Thời gian làm việc tiêu chuẩn là 8 giờ/ngày, từ Thứ Hai đến Thứ Sáu.')

In [ ]:
import time
import os
import psutil
import numpy as np
import chromadb
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import Chroma

embed = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")

print("Đang tính toán Brute-force Ground Truth để đo Recall...")
# Lấy embeddings của toàn bộ document (semantic_chunks từ Task 1)
doc_contents = [doc.page_content for doc in semantic_chunks]
doc_embeddings = embed.embed_documents(doc_contents)

# Danh sách query test
test_queries = [
    "Quy định về làm thêm giờ của FPT là gì?",
    "Chính sách nghỉ phép hàng năm",
    "Bảo mật thông tin dữ liệu công ty",
    "Trợ cấp ăn trưa và đi lại",
    "Quy trình xin nghỉ ốm dài ngày"
]
query_embeddings = embed.embed_documents(test_queries)

# Tìm Top 10 chuẩn xác bằng Brute-force (Cosine Similarity)
ground_truth_top10 = []
for q_emb in query_embeddings:
    sims = cosine_similarity([q_emb], doc_embeddings)[0]
    top10_idx = np.argsort(sims)[-10:][::-1]
    ground_truth_top10.append(set(top10_idx))


m_values = [16, 32, 64]
ef_construction_values = [100, 200, 400]
ef_search_values = [50, 100, 200]

results = []
process = psutil.Process(os.getpid())

print(f"\nBắt đầu benchmark trên {len(semantic_chunks)} chunks...")
print("-" * 105)
print(f"{'M':<4} | {'ef_con':<6} | {'ef_sea':<6} | {'Build (s)':<10} | {'Lat (ms)':<10} | {'Mem (MB)':<10} | {'Recall@10'}")
print("-" * 105)


for m in m_values:
    for ef_c in ef_construction_values:
        for ef_s in ef_search_values:
            
            # Đo RAM trước khi build
            mem_before = process.memory_info().rss / (1024 * 1024)
            
            # Metadata HNSW (Đã sửa lỗi chính tả hnsw)
            collection_metadata = {
                "hnsw:space": "cosine",
                "hnsw:M": m,
                "hnsw:construction_ef": ef_c,
                "hnsw:search_ef": ef_s
            }
            
            client = chromadb.EphemeralClient()
            collection_name = f"bench_{m}_{ef_c}_{ef_s}"
            
            # 3.1. Đo Build Time
            t0 = time.perf_counter()
            vs = Chroma.from_documents(
                documents=semantic_chunks,
                embedding=embed,
                collection_name=collection_name,
                client=client,
                collection_metadata=collection_metadata
            )
            build_time = time.perf_counter() - t0
            
            # Đo RAM sau khi build
            mem_used = max(0.0, (process.memory_info().rss / (1024 * 1024)) - mem_before)
            
            # 3.2. Đo Query Latency & Recall
            latencies = []
            recalls = []
            
            for i, query in enumerate(test_queries):
                # Warm up
                vs.similarity_search(query, k=10)
                
                # Đo Latency
                t_q = time.perf_counter()
                docs = vs.similarity_search(query, k=10)
                latencies.append(time.perf_counter() - t_q)
                
                # Tính Recall@10 bằng cách so khớp nội dung
                retrieved_idx = set()
                for d in docs:
                    for idx, raw_doc in enumerate(semantic_chunks):
                        if d.page_content == raw_doc.page_content:
                            retrieved_idx.add(idx)
                            break
                
                hits = len(retrieved_idx.intersection(ground_truth_top10[i]))
                recalls.append(hits / 10.0)
            
            avg_latency_ms = (sum(latencies) / len(latencies)) * 1000
            avg_recall = sum(recalls) / len(recalls)
            
            # Print kết quả dòng hiện tại
            print(f"{m:<4} | {ef_c:<6} | {ef_s:<6} | {build_time:<10.4f} | {avg_latency_ms:<10.2f} | {mem_used:<10.2f} | {avg_recall:.2%}")
            
            results.append({
                "M": m,
                "ef_construction": ef_c,
                "ef_search": ef_s,
                "Build Time (s)": round(build_time, 4),
                "Avg Latency (ms)": round(avg_latency_ms, 2),
                "Memory Usage (MB)": round(mem_used, 2),
                "Recall@10": round(avg_recall, 4)
            })
            
            # Xóa collection để giải phóng RAM cho vòng lặp sau
            vs.delete_collection()


/tmp/ipykernel_52078/2465132148.py:9: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14234.38it/s]


Đang tính toán Brute-force Ground Truth để đo Recall...

Bắt đầu benchmark trên 12 chunks...
---------------------------------------------------------------------------------------------------------
M    | ef_con | ef_sea | Build (s)  | Lat (ms)   | Mem (MB)   | Recall@10
---------------------------------------------------------------------------------------------------------
16   | 100    | 50     | 0.0251     | 3.84       | 20.99      | 96.00%
16   | 100    | 100    | 0.0242     | 3.86       | 2.85       | 96.00%
16   | 100    | 200    | 0.0251     | 3.68       | 2.76       | 96.00%
16   | 200    | 50     | 0.0240     | 3.66       | 2.70       | 96.00%
16   | 200    | 100    | 0.0258     | 3.71       | 2.66       | 96.00%
16   | 200    | 200    | 0.0267     | 3.81       | 2.68       | 96.00%
16   | 400    | 50     | 0.0246     | 3.80       | 2.61       | 96.00%
16   | 400    | 100    | 0.0246     | 3.89       | 2.64       | 96.00%
16   | 400    | 200    | 0.0250     | 3.58       | 2.

Task 3: End-to-End RAG Pipeline

In [13]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [17]:

llm = ChatOllama(model="llama3.1", temperature=0) #

optimized_vs = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embed,
    collection_name="optimized_rag",
    collection_metadata={
        "hnsw:space": "cosine",
        "hnsw:M": 32,
        "hnsw:construction_ef": 200,
        "hnsw:search_ef": 100
    } #
)
optimized_retriever = optimized_vs.as_retriever(search_kwargs={"k": 5})

rag_prompt = ChatPromptTemplate.from_template("""
Bạn là trợ lý HR chuyên nghiệp. Hãy trả lời câu hỏi dựa trên ngữ cảnh dưới đây.
Nếu không biết, hãy nói không biết, đừng tự bịa câu trả lời.

Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời:"""
)

optimized_chain = (
    {"context": optimized_retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()

)
def answer_question(question):
    return optimized_chain.invoke(question)

In [18]:
question="Chính sách nghỉ phép năm cho nhân viên thâm niên trên 5 năm là gì?"
answer = answer_question(question)
print(f"\nCâu hỏi: {question}")
print(f"Trả lời: {answer}")


Câu hỏi: Chính sách nghỉ phép năm cho nhân viên thâm niên trên 5 năm là gì?
Trả lời: Chính sách nghỉ phép năm cho nhân viên thâm niên trên 5 năm được quy định tại Document metadata={'source': 'doc2.txt_chunk_1'} và page_content='Nhân viên có thâm niên từ 5 năm trở lên được cộng thêm 1 ngày phép/năm. * Đối với Thực tập sinh: Thực tập sinh không có ngày phép năm có hưởng lương, nhưng được phép xin nghỉ không lương tối đa 3 ngày/tháng.'

Cụ thể, nhân viên có thâm niên từ 5 năm trở lên sẽ được cộng thêm 1 ngày phép/năm.
